# AWS Glue Studio Notebook
##### You are now running a AWS Glue Studio notebook; To start using your notebook you need to start an AWS Glue Interactive Session.


#### Optional: Run this cell to see available notebook commands ("magics").


In [1]:
# %additional_python_modules sagemaker_pyspark
# %extra_jars "s3://textclassificationmldemo-model-archiving-us-east-1-2667/models/model-a/java-jars/sagemaker-spark.jar"

#### Testing Additional Modules


In [2]:
# from sagemaker_pyspark import SageMakerModel
# print("SageMaker PySpark imported successfully!")

SageMaker PySpark imported successfully!


####  Run this cell to set up and start your interactive session.


In [1]:
%idle_timeout 300
%glue_version 5.0
%worker_type G.1X
%number_of_workers 4

import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job

import json
import boto3
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, explode, udf
from pyspark.sql.types import StringType
  
    
# Initialize Spark session
spark = SparkSession.builder.appName("GlueNotebookTextClassification").getOrCreate()

# sc = SparkContext.getOrCreate()
# glueContext = GlueContext(sc)
# spark = glueContext.spark_session
# job = Job(glueContext)

Welcome to the Glue Interactive Sessions Kernel
For more information on available magic commands, please type %help in any new cell.

Please view our Getting Started page to access the most up-to-date information on the Interactive Sessions kernel: https://docs.aws.amazon.com/glue/latest/dg/interactive-sessions.html
Installed kernel version: 1.0.7 
Current idle_timeout is None minutes.
idle_timeout has been set to 300 minutes.
Setting Glue version to: 5.0
Previous worker type: None
Setting new worker type to: G.1X
Previous number of workers: None
Setting new number of workers to: 4
Trying to create a Glue session for the kernel.
Session Type: glueetl
Worker Type: G.1X
Number of Workers: 4
Idle Timeout: 300
Session ID: fabc2e84-376d-49e2-8487-223a066c3bd2
Applying the following default arguments:
--glue_kernel_version 1.0.7
--enable-glue-datacatalog true
Waiting for session fabc2e84-376d-49e2-8487-223a066c3bd2 to get into ready status...
Session fabc2e84-376d-49e2-8487-223a066c3bd2 has 

### Read Input JSON from S3

In [2]:
# Read JSON Data from S3
json_path = "s3://textclassificationmldemo-model-archiving-us-east-1-2667/models/model-a/input/input_data.json"
df = spark.read.option("multiLine", True).json(json_path)

# Extract the "TestData" field
df_exploded = df.select(explode(col("TestData")).alias("test_data"))

# Extract sentences for classification
sentences_df = df_exploded.select(col("test_data.request.sentence").alias("sentence"))

print("Input Sentences for Classification:")
sentences_df.show(truncate=False)

Input Sentences for Classification:
+---------------------------------------------------------------------------------------+
|sentence                                                                               |
+---------------------------------------------------------------------------------------+
|The new president has called for an emergency conference for international cooperation.|
|Baseball is one of the most popular sports in the United States.                       |
|Stock investing has higher returns in the long run.                                    |
|The development of science accelerated the development of mankind.                     |
+---------------------------------------------------------------------------------------+


#### Define SageMaker Model Inference


In [7]:
# Define SageMaker Inference UDF (Boto3 Client Initialized Inside the UDF)
def sagemaker_inference(sentence):
    try:
        # Initialize Boto3 inside UDF (Fixing PicklingError)
        sagemaker_runtime = boto3.client("sagemaker-runtime")

        # Send single sentence in JSON format
        payload = json.dumps({"sentence": sentence})

        # Invoke the SageMaker endpoint
        response = sagemaker_runtime.invoke_endpoint(
            EndpointName="TextClassificationMLDemo-TextClassification-Endpoint",
            ContentType="application/json",
            Body=payload
        )

        # Parse response
        result = json.loads(response["Body"].read().decode("utf-8"))
        return str(result.get("label", "unknown"))  # Convert to string for PySpark compatibility

    except Exception as e:
        return f"Error: {str(e)}"

# Register UDF in PySpark (Fixing Serialization Issue)
sagemaker_udf = udf(sagemaker_inference, StringType())  # Ensure UDF returns StringType()

# Apply UDF for Classification
results_df = sentences_df.withColumn("prediction", sagemaker_udf(col("sentence")))

# Show Results
print("NEWS CATEGORIES")
print("1: 'World'\n2: 'Sports'\n3: 'Business'\n4: 'Sci/Tech'")
print("Classified Results:")
results_df.show(truncate=False)

# Write Predictions to S3
output_path = "s3://textclassificationmldemo-model-archiving-us-east-1-2667/models/model-a/output/predictions/"
results_df.write.mode("overwrite").option("compression", "gzip").json(output_path)

print(f"Predictions saved to: {output_path}")


NEWS CATEGORIES
1: 'World'
2: 'Sports'
3: 'Business'
4: 'Sci/Tech'
Classified Results:
+---------------------------------------------------------------------------------------+----------+
|sentence                                                                               |prediction|
+---------------------------------------------------------------------------------------+----------+
|The new president has called for an emergency conference for international cooperation.|1         |
|Baseball is one of the most popular sports in the United States.                       |2         |
|Stock investing has higher returns in the long run.                                    |3         |
|The development of science accelerated the development of mankind.                     |4         |
+---------------------------------------------------------------------------------------+----------+

Predictions saved to: s3://textclassificationmldemo-model-archiving-us-east-1-2667/models/model-a/output

Input Sentences for Classification:
+---------------------------------------------------------------------------------------+
|sentence                                                                               |
+---------------------------------------------------------------------------------------+
|The new president has called for an emergency conference for international cooperation.|
|Baseball is one of the most popular sports in the United States.                       |
|Stock investing has higher returns in the long run.                                    |
|The development of science accelerated the development of mankind.                     |
+---------------------------------------------------------------------------------------+

NEWS CATEGORIES
1: 'World'
2: 'Sports'
3: 'Business'
4: 'Sci/Tech'
Classified Results:
+---------------------------------------------------------------------------------------+-------------------------------------------------------------------